In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

notebook_path = Path().absolute()
sys.path.append(str(notebook_path.parent))

In [3]:
import torch
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from neural_controllers import NeuralController
from utils import newton_dataset, newton_dataset_new, emotion_dataset

SEED = 0

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

In [5]:
custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_name='llama_3_8b_it'

language_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    cache_dir=custom_cache_dir,
)

use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    use_fast=use_fast_tokenizer, 
    padding_side="left", 
    legacy=False,
)

# tokenizer.pad_token_id = 0 if tokenizer.pad_token_id is None else tokenizer.pad_token_id
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
controller = NeuralController(
    language_model,
    tokenizer,
    rfm_iters=8,
    batch_size=4,
    control_method='rfm'
)

n_components: 5
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 4
M_batch_size         : 2048
n_components         : 5



In [7]:
concept_types = ["Cam", "Isaac"]
data_dir = "../data/prefixed_newton"
dataset = newton_dataset_new(data_dir, controller, concept_types, samples=350, seed=SEED)

# concept_types = ["love", "hate"]
# data_dir = "../data/emotions"
# dataset = emotion_dataset(data_dir, controller, concept_types, samples=290)

Train data: 700
Test data: 224
Train data: 700
Test data: 224


In [8]:
# print(len(dataset["love"]["train"]["inputs"]))
# print(len(dataset["love"]["train"]["labels"]))

# print(dataset["love"]["train"]["inputs"][0])
# print(dataset["love"]["train"]["inputs"][1])
# print(dataset["love"]["train"]["labels"][0])
# print(dataset["love"]["train"]["inputs"][2])
# print(dataset["love"]["train"]["inputs"][3])
# print(dataset["love"]["train"]["labels"][1])

In [9]:
def find_lingering_forward_hooks(model):
    lingering_hooks = []
    for name, module in model.named_modules():
        if module._forward_hooks:
            hook_info = (name, 'forward')
            lingering_hooks.append(hook_info)
            print(f"Warning: Found {len(module._forward_hooks)} lingering 'forward' hook(s) on module: {name}")

        if module._forward_pre_hooks:
            hook_info = (name, 'forward_pre')
            lingering_hooks.append(hook_info)
            print(f"Warning: Found {len(module._forward_pre_hooks)} lingering 'forward_pre' hook(s) on module: {name}")
            
    if not lingering_hooks:
        print("Success: No lingering forward hooks found in the model.")
        
    return lingering_hooks

In [10]:
# rfm_iters = 16
rfm_iters = 8
# batch_size = 8
batch_size = 4
n_components = 300
# n_components = 5
# energy = 0.98

In [11]:
# controllers = {}

# for concept_type in tqdm(concept_types):
    
#     other_type = [k for k in concept_types if k != concept_type][0]
    
#     train_data = dataset[concept_type]['train']
#     test_data = dataset[concept_type]['test']
    
#     controller = NeuralController(
#         language_model,
#         tokenizer,
#         rfm_iters=rfm_iters,
#         batch_size=batch_size,
#         control_method='rfm',
#         n_components=n_components,
#         # energy=energy,
#     )
    
#     controller.compute_directions(train_data['inputs'], train_data['labels'])
    
#     controllers[concept_type] = controller

In [12]:
component_idx = 1
t_ite = 15

# anti = "yes"
anti = "no"

# control_coef=1.0
control_coef=0.4
# control_coef=0.9

# base_path = "../directions/stable"
base_path = "../directions/stable_base_config"
# path = f"{base_path}/isaac_cam_{n_components}_orig_ite_d{component_idx}_{SEED}"
path = f"{base_path}/isaac_cam_{n_components}_orig_ite_c{control_coef}_{SEED}"

In [13]:
# controllers = {}

print("Starting training:")

for concept_type in tqdm(concept_types):
    
    other_type = [k for k in concept_types if k != concept_type][0]
    
    train_data = dataset[concept_type]['train']
    test_data = dataset[concept_type]['test']
    
    controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=rfm_iters,
        batch_size=batch_size,
        control_method='rfm',
        n_components=n_components,
        # energy=energy,
    )

    # hidden_layers = None
    # hidden_layers = [-1, -2, -3, -4, -5]

    
    # controller.compute_directions(train_data['inputs'], train_data['labels'])

    controller.compute_directions_ite(
        train_data['inputs'], 
        train_data['labels'],
        # hidden_layers=hidden_layers,
        control_coef=control_coef,
        component_idx=component_idx,
        anti=anti,
    )

    # controller.compute_directions_ite_all(
    #     train_data['inputs'], 
    #     train_data['labels'],
    #     # hidden_layers=hidden_layers,
    #     control_coef=1.0,
    #     component_idx=component_idx,
    #     t_ite=t_ite,
    # )
    
    # controllers[concept_type] = controller



    # base_path = "/scratch/bbjr/skarmakar/dementia/ckpt"
    # base_path = "../directions/stable"
    # path = f"{base_path}/isaac_cam_{n_components}_itea_d{component_idx}_{SEED}"
    
    os.makedirs(path, exist_ok=True)
    controller.save(concept=f'{concept_type}', model_name='llama_3_8b_it', path=path)
    # break

Starting training:


  0%|          | 0/2 [00:00<?, ?it/s]

n_components: 300
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 4
M_batch_size         : 2048
n_components         : 300

Tuning metric: auc
Getting activations from forward passes


100%|██████████| 140/140 [00:58<00:00,  2.39it/s]


Getting activations from forward passes


...

Layers hooked dict_keys([-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2])
Tuning metric: auc
Getting activations from forward passes


100%|██████████| 140/140 [01:13<00:00,  1.90it/s]


Getting activations from forward passes


100%|██████████| 2/2 [1:39:38<00:00, 2989.17s/it]

Layers hooked dict_keys([-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1])


In [14]:
lh = find_lingering_forward_hooks(language_model)

Success: No lingering forward hooks found in the model.


In [15]:
# print(controllers["Cam"].directions.keys())
# print(len(controllers["Cam"].directions[-1]))
# print(controllers["Cam"].directions[-1][0].shape)

In [16]:
# /u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/xrfm/rfm_src/recursive_feature_machine.py

In [17]:
# print(controllers["Cam"].directions.keys())
# print(controllers["Cam"].directions[-31].shape)
# print(controllers["Isaac"].directions.keys())
# print(controllers["Isaac"].directions[-31].shape)

In [18]:
# base_path = "/scratch/bbjr/skarmakar/dementia/ckpt"
# # base_path = "../directions/stable"

# # path = f"../directions/stable/isaac_cam_{n_components}_ite"
# # path = f"../directions/stable/isaac_cam_{n_components}_3_{SEED}"
# # path = f"../directions/stable/emotions_{n_components}_2"

# # path = f"../directions/stable/isaac_cam_{n_components}_itea_d{component_idx}_{SEED}"

# path = f"{base_path}/isaac_cam_{n_components}_itea_d{component_idx}_{SEED}"

# os.makedirs(path, exist_ok=True)

# for concept_type in concept_types:
#     controller = controllers[concept_type]
#     # other_type = [k for k in concept_types if k!=concept_type][0]
    
#     controller.save(concept=f'{concept_type}', model_name='llama_3_8b_it', path=path)